# Урок 1. Первая модель: предсказываем выживание

Идея: модель смотрит на примеры с ответами (`fit`), находит правило и угадывает ответы для новых людей (`predict`).

## 0. Подготовка

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

df = pd.read_csv("../data/titanic.csv")
df["Sex_num"] = df["Sex"].map({"male": 0, "female": 1})   # модель понимает только числа

features4 = ["Pclass", "Sex_num", "Age", "Fare"]
features6 = features4 + ["SibSp", "Parch"]
y = df["Survived"]                                          # правильные ответы

## 1. Первая модель

Делим 891 пассажира: 712 на учёбу, 179 на экзамен. Модель учится и угадывает.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42, stratify=y
)

# Модель = заполнить пропуски возраста медианой + логистическая регрессия
model = make_pipeline(SimpleImputer(strategy="median"),
                      LogisticRegression(max_iter=1000))
model.fit(X_train[features4], y_train)        # учится на 712
pred = model.predict(X_test[features4])        # угадывает 179

baseline = (y_test == 0).mean()               # ленивый способ: «все погибли»
print(f"Ленивый способ: {baseline:.3f}")
print(f"Модель:         {accuracy_score(y_test, pred):.3f}")

Ленивый способ: 0.615
Модель:         0.788


**Вывод:** модель угадала 78.8%, а ленивый способ 61.5%. Значит, модель действительно что-то выучила.

## 2. Что выучила модель: коэффициенты («баллы»)

Плюс = признак повышает шанс выжить, минус = понижает.

In [3]:
coefs = model[-1].coef_[0]          # model[-1] = последний шаг pipeline, сама регрессия
pd.Series(coefs, index=features4).round(3)

Pclass    -1.117
Sex_num    2.509
Age       -0.034
Fare       0.001

**Вывод:** женщина (+2.5) — шанс выше, номер класса (−1.1) — чем больше номер, тем шанс ниже, возраст (−0.03) — старше хуже, цена (+0.001) — дороже лучше.
Всё совпадает с выводами EDA. Размер коэффициентов сравнивать нельзя: признаки в разных единицах (годы, фунты, 0/1).

## 3. Простое правило «женщины выжили» против модели

In [4]:
rule = X_test["Sex_num"]            # ответ = пол: женщина 1, мужчина 0
print(f"Правило по полу: {accuracy_score(y_test, rule):.3f}")
print(f"Модель:          {accuracy_score(y_test, pred):.3f}")

Правило по полу: 0.777
Модель:          0.788


**Вывод:** правило всего чуть хуже модели (77.7% против 78.8%). Почти всё умение модели — это пол.

## 4. 4 признака против 6 на пяти разбиениях

Сравниваем всегда на **одном и том же** тесте (одинаковый `random_state`).

In [5]:
rows = []
for rs in [0, 1, 2, 3, 4]:
    X_train, X_test, y_train, y_test = train_test_split(
        df, y, test_size=0.2, random_state=rs, stratify=y
    )
    row = {"random_state": rs}
    for name, feats in [("4 признака", features4), ("6 признаков", features6)]:
        m = make_pipeline(SimpleImputer(strategy="median"),
                          LogisticRegression(max_iter=1000))
        m.fit(X_train[feats], y_train)
        row[name] = accuracy_score(y_test, m.predict(X_test[feats]))
    rows.append(row)

results = pd.DataFrame(rows)
results["разница"] = results["6 признаков"] - results["4 признака"]
results.round(3)

,random_state,4 признака,6 признаков,разница
0,0,0.788,0.771,-0.017
1,1,0.777,0.782,0.006
2,2,0.765,0.777,0.011
3,3,0.816,0.821,0.006
4,4,0.782,0.821,0.039


In [6]:
results[["4 признака", "6 признаков", "разница"]].mean().round(3)

4 признака     0.785
6 признаков    0.794
разница        0.009

**Вывод:**
- Разный `random_state` = разные 179 человек в тесте, поэтому точность прыгает (от 0.765 до 0.816).
- В среднем 6 признаков чуть лучше (0.794 против 0.785), но на `random_state=0` стало хуже, а на 4 намного лучше.
- Разница меньше, чем разброс между разбиениями, поэтому уверенно сказать, что `SibSp` и `Parch` помогают, нельзя.